# MobAI Warehouse Forecasting - COMPLETE DEEP DIVE

## 🎯 THE CORE OBJECTIVE

**Predict tomorrow's demand** → Generate **Preparation Orders** one day ahead → Know which products to move from STORAGE to PICKING locations.

---

## 🛠️ STEP 1: DATA EXTRACTION & JOINING

We start by loading the demand history and enriching it with product attributes. 

> [!NOTE]
> We are using the preprocessed data from `ai/data/raw/products_for_ts_grouped.csv` which already contains the joined attributes and is aggregated by day.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Load preprocessed grouped data
df = pd.read_csv('../data/raw/products_for_ts_grouped.csv')

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

print(f"Total records: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique products: {df['id_produit'].nunique()}")

df.head()

## 🛠️ STEP 2: TEMPORAL FEATURE ENGINEERING

Demand often follows strong weekly or monthly patterns. We extract these features to help our models understand seasonality.

In [ ]:
def add_temporal_features(df):
    """
    Extract temporal features that influence demand
    """
    df = df.copy()
    
    # Extract date components
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day_of_month'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek  # Monday=0, Sunday=6
    df['week_of_year'] = df['date'].dt.isocalendar().week
    
    # Binary flags
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = (df['day_of_month'] <= 7).astype(int)
    df['is_month_end'] = (df['day_of_month'] >= 24).astype(int)
    
    return df

df_with_features = add_temporal_features(df)
print("Temporal features added.")
df_with_features[['date', 'day_of_week', 'is_weekend', 'is_month_start', 'is_month_end']].head()

## 🛠️ STEP 3: LAGGED & ROLLING FEATURES

Tomorrow's demand is often related to what happened yesterday or last week. We create rolling averages to capture trends.

In [ ]:
def create_features(df):
    df = df.sort_values(['id_produit', 'date']).copy()
    
    # Lags
    for lag in [1, 7, 30]:
        df[f'lag_{lag}d'] = df.groupby('id_produit')['quantite_demande'].shift(lag)
    
    # Rolling means
    for window in [7, 30]:
        df[f'rolling_mean_{window}d'] = df.groupby('id_produit')['quantite_demande'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
        
    return df.fillna(0)

df_final = create_features(df_with_features)
print("Lagged and rolling features created.")
df_final.head()

## 🛠️ STEP 4: PRODUCT SEGMENTATION

Not all products behave the same. Some are ordered every day in large quantities, while others are rare. We segment them to apply the best model for each.

In [ ]:
def segment_products(df):
    # Calculate stats per product
    stats = df.groupby('id_produit').agg({
        'quantite_demande': ['count', 'mean', 'std'],
        'date': lambda x: (x.max() - x.min()).days
    }).reset_index()
    
    stats.columns = ['id_produit', 'order_days', 'avg_qty', 'std_qty', 'days_span']
    stats['frequency'] = stats['order_days'] / (stats['days_span'] + 1)
    
    def classify(row):
        if row['frequency'] > 0.8 and row['avg_qty'] > 500: return 'A_HIGH_FREQ_VOL'
        if row['frequency'] > 0.5: return 'B_MEDIUM_FREQ'
        return 'C_LOW_FREQ'
    
    stats['segment'] = stats.apply(classify, axis=1)
    return stats

product_segments = segment_products(df_final)
df_final = df_final.merge(product_segments[['id_produit', 'segment']], on='id_produit')
print("Product segmentation complete.")
print(df_final['segment'].value_counts())

## 🛠️ STEP 5: FORECASTING MODELS

We define our models, ranging from simple averages to Machine Learning.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def get_naive_forecast(history, window=7):
    return history.tail(window).mean()

def train_rf_model(train_df, features):
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    model.fit(train_df[features], train_df['quantite_demande'])
    return model

print("Model functions defined.")

## 🛠️ STEP 6: ORCHESTRATION - GENERATING THE ORDER

We combine the models to generate the final **Preparation Order** for a target date.

In [ ]:
def generate_order(df, target_date):
    # Sample logic for a single product to demonstrate
    # In production, this would loop through segments and use mapped models
    history = df[df['date'] < target_date]
    
    # Naive example
    forecast = history.groupby('id_produit')['quantite_demande'].mean().reset_index()
    forecast.columns = ['id_produit', 'forecasted_quantity']
    
    return forecast

target_date = df['date'].max()
prep_order = generate_order(df_final, target_date)
print(f"Generated Preparation Order for {target_date.date()} with {len(prep_order)} items.")

## 📊 STEP 7: EVALUATION & VALIDATION

Finally, we evaluate our forecast against actual demand to measure accuracy (MAPE, Bias, Service Level).

In [ ]:
from sklearn.metrics import mean_absolute_error

def evaluate(actual, forecast):
    mae = mean_absolute_error(actual, forecast)
    print(f"MAE: {mae:.2f} units")
    return mae

print("Evaluation logic ready.")